# Testing the Behavioural LSTM

This notebook lets you poke at the trained **`behavior_model.h5`** directly — the same model called by `fraud-service` over `POST /predict/behavioral`.

**Run it via:**
```bash
cd "Jiya project/TrackEasy-3.O/TrackEasy"
docker compose -f docker-compose.train.yml up --build   # opens Jupyter on http://localhost:8888
```
Then open this file at `Jiya project/TrackEasy-3.O/TrackEasy/fraud-service/ml/test_lstm.ipynb` in JupyterLab.

## Model summary (from training)

| Property | Value |
|---|---|
| Architecture | `LSTM(64) → Dropout(0.3) → Dense(32, relu) → Dense(1, sigmoid)` |
| Input shape | `(batch, 10, 1)` — 10 time-steps, single integer feature |
| Output | One probability ∈ [0, 1] (fraud confidence) |
| Training data | `behavioral_dataset.csv` — 2000 rows, but only **3 unique sequences** |
| Loss | binary_crossentropy |
| Loaded by | `ml/ml_service.py` at startup, served at `POST /predict/behavioral` |
| Used by | `fraud-service/fraudServer.js`: fires `+5` to riskScore when `prob > 0.85` |

### Event encoding (used everywhere)
```
1 = login         2 = add_to_cart    3 = remove_from_cart
4 = checkout      5 = payment_failed 6 = payment_success
0 = padding (used when fewer than 10 events available)
```

## 1. Setup

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

MODEL_PATH = 'behavior_model.h5'
DATA_PATH = 'behavioral_dataset.csv'

EVENT_NAMES = {0: 'pad', 1: 'login', 2: 'add_to_cart', 3: 'remove_from_cart',
               4: 'checkout_attempt', 5: 'payment_failed', 6: 'payment_success'}

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

## 2. The training dataset — *only 3 unique sequences*

Run the cell below and observe: 2000 rows, but `df.drop_duplicates()` returns only 3.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Total rows: {len(df)}')
print(f'Unique sequences: {len(df.drop_duplicates())}')
print('\nDistinct rows:')
df.drop_duplicates().reset_index(drop=True)

## 3. Helper — predict + pretty-print

Mirrors the exact pre-processing in [ml_service.py:122-133](../../../../README.md):
1. Pad/truncate to length 10 (zeros at the **front**)
2. Reshape to `(1, 10, 1)`
3. Forward pass → sigmoid

In [ ]:
def predict(seq, label=None):
    """Match the production pre-processing in ml_service.py."""
    seq = list(seq)
    if len(seq) < 10:
        seq = [0] * (10 - len(seq)) + seq
    else:
        seq = seq[:10]
    X = np.array(seq).reshape(1, 10, 1)
    prob = float(model.predict(X, verbose=0)[0][0])
    fired = prob > 0.85
    decoded = ' → '.join(EVENT_NAMES[v] for v in seq)
    label_str = f' [{label}]' if label else ''
    fired_str = ' FIRED (+5)' if fired else ''
    print(f'{seq}{label_str}')
    print(f'  decoded: {decoded}')
    print(f'  prob:    {prob:.6f}{fired_str}')
    print()
    return prob

## 4. Predict on the training templates

These are the *only* shapes the model has seen. Expect: normal ≈ 0, both bot patterns ≈ 1.

In [ ]:
predict([1, 2, 2, 3, 2, 4, 6, 2, 2, 2], label='Normal template')
predict([2, 2, 2, 2, 2, 2, 2, 2, 2, 2], label='Bot template — rapid add_to_cart')
predict([5, 5, 5, 5, 5, 5, 5, 5, 5, 5], label='Bot template — repeated payment_failed')

## 5. How robust is it to *partial* bot patterns?

Sweep how many `payment_failed` (5) events appear in an otherwise-normal sequence and watch the probability move.

In [ ]:
results = []
for n_failures in range(11):
    seq = [5] * n_failures + [2] * (10 - n_failures)
    X = np.array(seq).reshape(1, 10, 1)
    prob = float(model.predict(X, verbose=0)[0][0])
    results.append((n_failures, prob))

df_sweep = pd.DataFrame(results, columns=['payment_failed_count', 'fraud_probability'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_sweep['payment_failed_count'], df_sweep['fraud_probability'])
ax.axhline(0.85, color='red', linestyle='--', label='fire threshold (0.85)')
ax.set_xlabel('# of payment_failed events in 10-event window')
ax.set_ylabel('LSTM fraud probability')
ax.set_title('Sensitivity to repeated payment failures')
ax.legend()
plt.tight_layout()
plt.show()
df_sweep

## 6. Cold-start — what happens with very few events?

Brand-new users (and Alice right after we wiped her EventLog) only have 1-2 events. Production pre-processing zero-pads at the front, so the input becomes mostly `0`s.

What does the model say for short, padded sequences?

In [ ]:
predict([1], label='Just logged in (1 event)')
predict([1, 2], label='Login + 1 add_to_cart')
predict([1, 2, 2, 4], label='Login + 2 adds + checkout')
predict([1, 2, 2, 2, 4, 6], label='Realistic but no payment failures (6 events)')

## 7. Novel realistic sequences not in the training set

What about sequences that look like a real human — login, browse around, occasional remove, eventually checkout — but **don't match any training template exactly**?

In [ ]:
predict([1, 2, 3, 2, 2, 3, 2, 4, 6, 2], label='Picky shopper — adds and removes')
predict([1, 2, 2, 4, 5, 4, 6, 2, 2, 2], label='Failed once then succeeded')
predict([1, 2, 2, 4, 5, 4, 5, 4, 6, 2], label='Failed twice then succeeded')
predict([1, 2, 2, 4, 5, 5, 4, 6, 2, 2], label='Two consecutive failures then success')
predict([1, 4, 6, 1, 4, 6, 1, 4, 6, 1], label='Quick re-orders (subscription style)')

## 8. Compare two perfectly-mixed bot patterns — does order matter?

Both have 5 `payment_failed` and 5 `add_to_cart`. The LSTM is supposed to capture **temporal order**. Let's see whether shuffling actually changes the prediction.

In [ ]:
predict([5, 5, 5, 5, 5, 2, 2, 2, 2, 2], label='Failures then adds (block A)')
predict([2, 2, 2, 2, 2, 5, 5, 5, 5, 5], label='Adds then failures (block B)')
predict([5, 2, 5, 2, 5, 2, 5, 2, 5, 2], label='Strictly alternating')
predict([2, 5, 2, 5, 2, 5, 2, 5, 2, 5], label='Alternating, opposite phase')

## 9. Free-form playground

Edit `MY_SEQUENCE` and run the cell. Try things like:
- `[1, 2, 2, 4, 6]` — a clean short shopping flow
- `[2, 2, 2, 5, 5, 5, 4, 5, 5, 5]` — failures dominating
- `[1, 2, 3, 4, 5, 6, 1, 2, 3, 4]` — every event type once + repeats

In [ ]:
MY_SEQUENCE = [1, 2, 2, 4, 5, 5, 4, 6]
predict(MY_SEQUENCE, label='custom')

## 10. Probability surface — full 2-D sweep

For every combination of (n_payment_failed, n_add_to_cart) totalling 10, plot the fraud probability. Should reveal which corners of the input space the LSTM cares about.

In [ ]:
# Build a grid: rows=#failed (5s), cols=#add_to_cart (2s); fill remainder with 1s (login)
size = 11
grid = np.zeros((size, size))
for n5 in range(size):
    for n2 in range(size):
        if n5 + n2 > 10:
            grid[n5, n2] = np.nan
            continue
        seq = [5] * n5 + [2] * n2 + [1] * (10 - n5 - n2)
        X = np.array(seq).reshape(1, 10, 1)
        grid[n5, n2] = float(model.predict(X, verbose=0)[0][0])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid, origin='lower', cmap='Reds', vmin=0, vmax=1)
ax.set_xlabel('# add_to_cart events (2)')
ax.set_ylabel('# payment_failed events (5)')
ax.set_title('Fraud probability surface\n(remaining slots filled with login=1)')
plt.colorbar(im, ax=ax, label='fraud prob')
for n5 in range(size):
    for n2 in range(size):
        if not np.isnan(grid[n5, n2]):
            ax.text(n2, n5, f'{grid[n5, n2]:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if grid[n5, n2] > 0.5 else 'black')
plt.tight_layout()
plt.show()

## 11. What this tells us

After running the cells above you should see the same conclusions we reached during the audit:

1. **Template matching, not learning.** The 3 training rows give 3 sharp predictions. Everything else is interpolation between those 3 points.
2. **Domination by `5`s drives prob → 1.** That's why Alice's polluted EventLog (7 of 10 events were `payment_failed`) scored 99.99%.
3. **Cold-start sequences are unstable.** Heavily zero-padded inputs are not represented in the training set — the prediction depends on whatever the LSTM extrapolates.
4. **Order doesn't matter much** for sequences with the same composition (cell 8). The LSTM is essentially counting integers, not modelling temporal structure.
5. **Mixed sequences in cell 7** show the model is uncertain on realistic-but-novel inputs — usually probability < 0.5, never close to 0 (because they don't match the *normal* template either).

## To improve this model (TODO #C1 in the repo TODO.md)

- Generate diverse training sequences via random walks over the event vocabulary with realistic transition probabilities.
- Include borderline negatives (e.g. 1-2 failed payments interspersed with normal flow → labelled 0).
- Use a small Transformer with positional embeddings if temporal order actually matters for your fraud cases.
- Validate on held-out, **labelled real session data** — not synthetic templates.